# 영화 리뷰 워드 임베딩 (Word2Vec, FastText)
- gensim 라이브러리 사용 : pip install gensim
    - Word2Vec : models.Word2Vec
    - FastText : models.FastText

## 1. 데이터 준비
* 토큰화가 잘 되어 있는 filtered 데이터 사용

In [18]:
import pandas as pd
data_filename = './data/Korean_movie_reviews_2016_filtered.csv'
data_df = pd.read_csv(data_filename)
data_df.head()

,review,rate
0,아니 딴 그렇 비 비탄 총 대체 왜 들 온겨,7
1,진심 쓰레기 영화 만들 무서 알 쫄아 틀었 이건 뭐 웃 거리 없는 쓰레기 영화 임,1
2,역대 좀비 영화 가장 최고다 원작 만화 읽어 보려 영화 보고 결정 하려 감독 간츠 ...,10
3,온종일 불편한 피 범벅 일,6
4,답답함 극치 움직일 잇으 좀 움직여 어지간히 좀비 봣으 얼 타고 때려 잡 때 되 않냐,1


In [19]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 788189 entries, 0 to 788188
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  785448 non-null  object
 1   rate    788189 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 12.0+ MB


In [20]:
# 결측치 제거 (rate에 결측치)
# 결측치를 제거하지 않으면 review 데이터 추출 시 float으로 타입 변환됨
data_df.dropna(inplace=True)

In [21]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 785448 entries, 0 to 788188
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  785448 non-null  object
 1   rate    785448 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 18.0+ MB


In [22]:
# review만 모아서 review별 토큰 리스트로 변환
review_list = list(data_df.review)
review_list[:5]

['아니 딴 그렇 비 비탄 총 대체 왜 들 온겨',
 '진심 쓰레기 영화 만들 무서 알 쫄아 틀었 이건 뭐 웃 거리 없는 쓰레기 영화 임',
 '역대 좀비 영화 가장 최고다 원작 만화 읽어 보려 영화 보고 결정 하려 감독 간츠 실사 했 사람 거르려 그냥 봤 정말 흠잡 없는 최고 좀비 영화 잔인 거 싫어하지 참고 볼 만하 로미 인물 왜 그런 모르',
 '온종일 불편한 피 범벅 일',
 '답답함 극치 움직일 잇으 좀 움직여 어지간히 좀비 봣으 얼 타고 때려 잡 때 되 않냐']

In [23]:
type(review_list[0])

str

In [24]:
# 결측치 처리 시 ValueError 예외 발생
token_list = [review.split() for review in review_list]
token_list[:2]

[['아니', '딴', '그렇', '비', '비탄', '총', '대체', '왜', '들', '온겨'],
 ['진심',
  '쓰레기',
  '영화',
  '만들',
  '무서',
  '알',
  '쫄아',
  '틀었',
  '이건',
  '뭐',
  '웃',
  '거리',
  '없는',
  '쓰레기',
  '영화',
  '임']]

## 1. Word2Vec 활용 영화 리뷰 워드 임베딩
* https://radimrehurek.com/gensim/models/word2vec.html

### Skipgram, negative=10 인 경우

In [25]:
# Word2Vec 모델 생성 및 학습 : window=3, min_count=3
from gensim.models import Word2Vec
model_sg_n10 = Word2Vec(token_list, vector_size=100, window=3, min_count=3, sg=1, negative=10)

In [26]:
# 단어의 임베딩 벡터 확인
model_sg_n10.wv['이정재']

array([-5.62961288e-02,  1.95748657e-01, -1.21853732e-01, -7.34749436e-01,
       -2.31505230e-01, -1.93944007e-01,  4.45838392e-01,  2.29913801e-01,
        7.67369151e-01, -1.52228430e-01, -1.51610589e-02, -1.54735312e-01,
        3.83836292e-02,  1.40567839e-01, -1.61390841e-01, -1.65530756e-01,
       -1.46162957e-01,  1.89500660e-01, -4.43724766e-02, -1.03172295e-01,
        4.01985586e-01,  5.09858608e-01,  1.05096072e-01, -1.71322912e-01,
       -1.63235441e-01, -3.92209917e-01, -2.17046931e-01,  2.40833476e-01,
       -1.25630453e-01, -2.68553406e-01,  3.27730030e-02, -3.11103821e-01,
       -3.75038050e-02,  6.74818992e-04, -1.74857050e-01, -8.22752535e-01,
       -4.68877524e-01, -2.40530550e-01, -3.70291710e-01, -6.39567822e-02,
       -4.90380764e-01,  3.51505578e-01, -3.57541621e-01, -3.95422667e-01,
        2.99008023e-02, -3.17569673e-02,  1.77989830e-03, -1.89210951e-01,
       -2.39426255e-01, -1.77497432e-01, -8.80555660e-02, -5.60014188e-01,
        5.73997088e-02, -

In [27]:
# 단어의 임베딩 벡터 차원 확인
len(model_sg_n10.wv['이정재'])

100

In [28]:
# 두 단어 간 유사도 확인
wv = model_sg_n10.wv
wv.similarity('이정재', '정우성')

np.float32(0.74504954)

In [29]:
# 특정 단어와 유사한 단어 추출
wv.most_similar('이정재', topn=20)

[('이범수', 0.8182013630867004),
 ('송강호', 0.7896728515625),
 ('공유', 0.7873256206512451),
 ('김범수', 0.7856687307357788),
 ('김남길', 0.7468278408050537),
 ('정우성', 0.7450495362281799),
 ('리암', 0.7287557125091553),
 ('조재현', 0.720475435256958),
 ('박해일', 0.7196944355964661),
 ('김명민', 0.717136800289154),
 ('이병헌', 0.7160217761993408),
 ('박철민', 0.7119370698928833),
 ('곽도원', 0.7071056962013245),
 ('마동석', 0.707003653049469),
 ('이진욱', 0.7040680050849915),
 ('요한', 0.7040368318557739),
 ('유준상', 0.7023449540138245),
 ('김남일', 0.7006523013114929),
 ('김희원', 0.7002806067466736),
 ('이성민', 0.7000192999839783)]

In [32]:
print([word for word, _ in wv.most_similar('재밌', topn=50)])

['재미있', '재밌네', '잼남', '재밌었', '재밋음', '재밌어', '재밋엇어용', '재밋는듯', '재밋엇음', '재밋었습니', '잼슴', '재밋게봣습니', '재미있었', '재밌슴', '재밋었어', '재밋어용', '재밋었음', '재미있더', '재미있네', '쟈밋', '재밌아', '재밋네', '재밋엇어', '재밋엇', '재밋습니', '재밋게봣어', '재밌던', '재밌드', '엇', '재밋게봣', '존잼임', '재밌고', '재밋구', '재밌았', '잼난', '재밋었', '꿀잼임', '재밋어', '재밋습니당', '재밋습', '재미있겠', '재밋당', '재밌더', '재밋게봣네', '짱재밋어', '재밋게봄', '재밋네용', '더재밋', '개꿀잼', '재밋었네']


### Skipgram, negative=5 인 경우

In [33]:
# 모델 생성
model_sg_n5 = Word2Vec(token_list, vector_size=100, window=3, min_count=3, sg=1, negative=5)

In [35]:
# 특어 단어와 유사한 단어 추출 : 이정재
wv = model_sg_n5.wv
wv.most_similar('이정재', topn=20)

[('이범수', 0.8441994190216064),
 ('송강호', 0.7982925772666931),
 ('공유', 0.7767969965934753),
 ('김범수', 0.7662171721458435),
 ('조재현', 0.7367668747901917),
 ('정우성', 0.7319844365119934),
 ('이성민', 0.7306202054023743),
 ('박해일', 0.7241826057434082),
 ('김윤석', 0.7201212644577026),
 ('이병헌', 0.7191486358642578),
 ('곽도원', 0.7184240221977234),
 ('요한', 0.7151025533676147),
 ('김남길', 0.7111530900001526),
 ('주지훈', 0.7104809284210205),
 ('리암', 0.7053989768028259),
 ('김명민', 0.7022249102592468),
 ('김성균', 0.6999539732933044),
 ('정재형', 0.6984001994132996),
 ('송광호', 0.6937641501426697),
 ('작대기', 0.6934357285499573)]

In [36]:
# 특어 단어와 유사한 단어 추출 : 재밌
print([word for word, _ in wv.most_similar('재밌', topn=50)])

['재미있', '재밌네', '재밌어', '재밋음', '재밌었', '잼남', '재밋어용', '재밌아', '재밋엇', '잼슴', '재밌슴', '재밋었어', '재밋구', '재밋엇어용', '재밋습니', '재밋게봣어', '재미있었', '쟈밋', '재밋엇음', '재밌더', '재밋어', '재밋엇어', '재밋었습니', '존잼임', '재밋네', '재밋었음', '잼난', '꿀잼', '재밋게봣습니', '엇', '재미있네', '재밌고', '재밋네용', '재밋습', '재밋습니당', '재밋는듯', '꿀잼임', '재밌엇습니', '재밌던', '재밋었', '재밋게봣', '재밋당', '개꿀잼', '존잼', '번보', '재미있구', '재밋게봄', '짱재밋음', '재밌으', '재미있더']


### CBOW, negative=10 인 경우

In [37]:
model_cbow_n10 = Word2Vec(token_list, vector_size=100, window=3, min_count=3, sg=0, negative=10)

In [41]:
wv = model_cbow_n10.wv
wv.most_similar('이정재', topn=20)

[('이범수', 0.8063144683837891),
 ('공유', 0.7451112270355225),
 ('김윤석', 0.7431967854499817),
 ('조재현', 0.7211953401565552),
 ('이성민', 0.7175747752189636),
 ('주지훈', 0.7100709676742554),
 ('김범수', 0.7077839374542236),
 ('송강호', 0.6897832155227661),
 ('김남길', 0.6886869072914124),
 ('이진욱', 0.674924910068512),
 ('민호', 0.674058735370636),
 ('박해일', 0.6679700613021851),
 ('곽도원', 0.666641891002655),
 ('남자배우', 0.6648289561271667),
 ('송광호', 0.6528510451316833),
 ('김성오', 0.6499914526939392),
 ('엄지원', 0.6497495770454407),
 ('임시완', 0.6496558785438538),
 ('정우성', 0.6494315266609192),
 ('조정석', 0.6490120887756348)]

In [43]:
print([word for word, _ in wv.most_similar('재밌', topn=50)])

['재미있', '재밌네', '재밌어', '재밌었', '재밋음', '잼남', '재밌는', '재미있네', '재밋어', '재밌더', '재미있었', '재밌던', '재밋엇어', '재미있어', '재밋네', '재밋엇', '재밋', '재밋었어', '재미있던', '꿀잼', '재밌다', '재밌고', '재밋엇음', '재밋었', '재미있는', '재밌구', '재미있다', '잼', '재밋습니', '재밋었음', '재밌게', '재미있더', '웃김', '재밌으', '재밌겠', '재미있고', '개꿀잼', '존잼', '괜찮', '재미나', '재미있게', '웃겼', '무서웠', '재밋어용', '졸잼', '꿀잼임', '재미있겠', '멋있', '핵꿀잼', '재미없']


### CBOW, negative=5 인 경우

In [51]:
model_cbow_n5 = Word2Vec(token_list, vector_size=100, window=3, min_count=3, sg=0, negative=5)

In [52]:
wv = model_cbow_n10.wv
wv.most_similar('이정재', topn=20)

[('이범수', 0.8063144683837891),
 ('공유', 0.7451112270355225),
 ('김윤석', 0.7431967854499817),
 ('조재현', 0.7211953401565552),
 ('이성민', 0.7175747752189636),
 ('주지훈', 0.7100709676742554),
 ('김범수', 0.7077839374542236),
 ('송강호', 0.6897832155227661),
 ('김남길', 0.6886869072914124),
 ('이진욱', 0.674924910068512),
 ('민호', 0.674058735370636),
 ('박해일', 0.6679700613021851),
 ('곽도원', 0.666641891002655),
 ('남자배우', 0.6648289561271667),
 ('송광호', 0.6528510451316833),
 ('김성오', 0.6499914526939392),
 ('엄지원', 0.6497495770454407),
 ('임시완', 0.6496558785438538),
 ('정우성', 0.6494315266609192),
 ('조정석', 0.6490120887756348)]

In [53]:
print([word for word, _ in wv.most_similar('재밌', topn=50)])

['재미있', '재밌네', '재밌어', '재밌었', '재밋음', '잼남', '재밌는', '재미있네', '재밋어', '재밌더', '재미있었', '재밌던', '재밋엇어', '재미있어', '재밋네', '재밋엇', '재밋', '재밋었어', '재미있던', '꿀잼', '재밌다', '재밌고', '재밋엇음', '재밋었', '재미있는', '재밌구', '재미있다', '잼', '재밋습니', '재밋었음', '재밌게', '재미있더', '웃김', '재밌으', '재밌겠', '재미있고', '개꿀잼', '존잼', '괜찮', '재미나', '재미있게', '웃겼', '무서웠', '재밋어용', '졸잼', '꿀잼임', '재미있겠', '멋있', '핵꿀잼', '재미없']


### OOV(Out of Vocabulary) 문제

In [45]:
# corpus에 없는 단어 확인 : 우주평화 
'우주평화' in model_sg_n10.wv.key_to_index

False

In [44]:
# corpus에 없는 단어의 임베딩 벡터 확인 
model_sg_n10.wv['우주평화']

KeyError: "Key '우주평화' not present"

## 2. FastText 활용 영화 리뷰 워드 임베딩
* https://radimrehurek.com/gensim/models/fasttext.html

In [46]:
# FastText 모델 생성 및 학습
# window=3, min_count=3, min_n=2, max_n=2
from gensim.models import FastText
model = FastText(token_list, vector_size=100, window=3, min_count=3, sg=1, negative=10, min_n=2, max_n=2)
wv = model.wv

In [47]:
# 특정 단어와 유사한 단어 추출 : 이정재
wv.most_similar('이정재', topn=20)

[('이범수', 0.8429310321807861),
 ('정재', 0.8191202878952026),
 ('임성민', 0.818389892578125),
 ('공유', 0.8148197531700134),
 ('김범수', 0.8084936141967773),
 ('정재영', 0.8059480786323547),
 ('이진욱', 0.7922002077102661),
 ('송강호', 0.7866899967193604),
 ('임완', 0.784087061882019),
 ('박해일', 0.7789525389671326),
 ('정재형', 0.7788062691688538),
 ('김남일', 0.7787118554115295),
 ('김지민', 0.7756251096725464),
 ('김성균', 0.768108606338501),
 ('이정진', 0.7645375728607178),
 ('김철민', 0.7624986171722412),
 ('이성민', 0.7615309953689575),
 ('김영민', 0.7611760497093201),
 ('조재현', 0.7592501044273376),
 ('김성민', 0.75267094373703)]

In [48]:
# corpus에 없는 단어 확인 : 우주평화 
'우주평화' in wv.key_to_index

False

In [49]:
# corpus에 없는 단어의 임베딩 벡터 확인 
wv['우주평화']

array([ 0.25571114,  0.09311666,  0.07112224,  0.29961044, -0.14174806,
       -0.21433175, -0.20876542,  0.73769176,  0.41337052,  0.1405311 ,
       -0.06169453,  0.09407298, -0.11599269,  0.3673973 , -0.2422247 ,
       -0.27446973,  0.13814807, -0.1726444 ,  0.06933452, -0.03022698,
        0.20864339, -0.10117419,  0.31032795, -0.0685572 , -0.1694301 ,
       -0.18055919, -0.38398015, -0.27636176, -0.15313956, -0.1502376 ,
        0.20890884, -0.347205  ,  0.20817304, -0.49324933, -0.05980337,
        0.16314706,  0.14011736,  0.25989923, -0.49642387, -0.02109304,
       -0.08730559,  0.08063383, -0.12373648,  0.08324539, -0.08934108,
        0.21416886, -0.18157859,  0.10267887,  0.08344104,  0.14227697,
        0.41244859,  0.19214411, -0.15298013, -0.06883411, -0.19253767,
       -0.03501654,  0.39643064,  0.24888971,  0.18203771,  0.1889961 ,
       -0.09444086,  0.39510387, -0.5195439 , -0.10304751, -0.18018779,
       -0.3702958 , -0.07084458, -0.2578186 , -0.04119008,  0.16

In [50]:
# corpus에 없는 단어와 유사한 단어추출 
wv.most_similar('우주평화', topn=20)

[('우주', 0.8204562664031982),
 ('평화', 0.8201935887336731),
 ('우주비행사', 0.8026214838027954),
 ('무궁화', 0.7962223291397095),
 ('우장', 0.793121337890625),
 ('투명인간', 0.7877389192581177),
 ('우방', 0.787716805934906),
 ('산업화', 0.7783003449440002),
 ('우주인', 0.7780728936195374),
 ('우주선', 0.7757300138473511),
 ('우화', 0.7732986211776733),
 ('예루살렘', 0.7692814469337463),
 ('산화', 0.7689490914344788),
 ('회색곰', 0.7681628465652466),
 ('경복궁', 0.7681043148040771),
 ('지구대', 0.7674888968467712),
 ('장래희망', 0.7673658132553101),
 ('우주여행', 0.7660335302352905),
 ('지옥도', 0.7655825614929199),
 ('파릇파릇', 0.764410138130188)]